# 04 · Supplier scorecard

**Question Q5.** Which sellers are the best and the worst within their category, and where should procurement act first?

**Design decisions carried from notebooks 02 and 03**

| Metric | In the score? | Why |
| --- | --- | --- |
| Dispatch on time (vs shipping limit) | Yes, highest weight | Controlled by the seller, persistent across years, independent of distance |
| Dispatch lead time p90 | Yes | Predictability, not only the average |
| Review score (Bayesian shrinkage) | Yes | Perceived quality; shrinkage stops a seller with 3 five-star reviews from leading |
| Share of 1-2 star reviews | Yes, low weight | Tail of dissatisfaction |
| Cancellation rate | Yes, weight set by its distribution below | Average is 0.4%, so it may barely discriminate |
| Freight / item price | Yes | Total cost of ownership: sourcing from a distant seller costs more, even when delays are not the seller's fault |
| Delivered to customer on time | **No** | About 72% of customer delays start in the logistics leg (notebook 03) |

**Method.** Sellers with 30+ valid orders are eligible. Each metric becomes a 0-100 percentile within the seller's main category (global percentile when the category has fewer than 5 eligible sellers), with 100 = best. The score is the weighted average of the percentiles.

In [13]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.stats import spearmanr

from src.data import q
from src.style import C, MONO, finish, save

pd.set_option("display.float_format", "{:,.3f}".format)
MIN_ORDERS = 30

## 1 · Eligibility

In [14]:
sc = q("""SELECT b.*, d.main_category, d.seller_state
          FROM mart_seller_scorecard_base b JOIN dim_sellers d USING (seller_id)""")
for col in ["spend_brl", "freight_brl"]:
    sc[col] = sc[col].astype(float)
# raw rate renamed so it cannot collide with its percentile column (pct_negative_reviews)
sc = sc.rename(columns={"pct_negative_reviews": "negative_review_rate"})
sc["eligible"] = sc["n_orders"] >= MIN_ORDERS
el = sc[sc["eligible"]].copy()
coverage = el["spend_brl"].sum() / sc["spend_brl"].sum()
print(f"Sellers in window:   {len(sc):,}")
print(f"Eligible (30+):      {len(el):,} ({len(el) / len(sc):.1%} of sellers, {coverage:.1%} of spend)")

Sellers in window:   3,068
Eligible (30+):      633 (20.6% of sellers, 77.1% of spend)


## 2 · Does cancellation discriminate?
If most eligible sellers have zero cancellations, the percentile of this metric is decided by one or two events, which is noise. The rule is written before looking at the result: **if more than 50% of eligible sellers have zero cancellations, cancellation weight drops from 15% to 5%** and the 10 points move to dispatch (5) and review (5).

In [15]:
zero_cancel = (el["cancel_rate"] == 0).mean()
print(f"Eligible sellers with zero cancellations: {zero_cancel:.1%}")
print(el["cancel_rate"].describe(percentiles=[.5, .75, .9, .95]).to_string())

ORDER = ["dispatch_on_time", "lead_p90", "review_bayes", "negative_reviews", "cancel", "freight_ratio"]
if zero_cancel > 0.50:
    W = dict(zip(ORDER, [.35, .10, .30, .05, .05, .15]))
    print("\nRule applied: cancellation weight 5%")
else:
    W = dict(zip(ORDER, [.30, .10, .25, .05, .15, .15]))
    print("\nRule not triggered: original weights kept")
assert abs(sum(W.values()) - 1) < 1e-9
pd.Series(W, name="weight")

Eligible sellers with zero cancellations: 77.3%
count   633.000
mean      0.004
std       0.009
min       0.000
50%       0.000
75%       0.000
90%       0.015
95%       0.024
max       0.071

Rule applied: cancellation weight 5%


dispatch_on_time   0.350
lead_p90           0.100
review_bayes       0.300
negative_reviews   0.050
cancel             0.050
freight_ratio      0.150
Name: weight, dtype: float64

## 3 · Review shrinkage
`review_bayes = (n x seller mean + m x global mean) / (n + m)`. The value of m decides how much evidence a seller needs to move away from the average; it is tested at 10, 20 and 40.

In [16]:
g = (el["review_mean"] * el["review_n"]).sum() / el["review_n"].sum()

def bayes(m):
    return (el["review_n"] * el["review_mean"].fillna(g) + m * g) / (el["review_n"] + m)

top = {m: set(el.assign(r=bayes(m)).nlargest(20, "r")["seller_id"]) for m in [10, 20, 40]}
print(f"Global mean review (order-weighted): {g:.2f}")
print(f"Top 20 by review, m=10 vs m=20 overlap: {len(top[10] & top[20]) / 20:.0%}")
print(f"Top 20 by review, m=40 vs m=20 overlap: {len(top[40] & top[20]) / 20:.0%}")
M = 20
el["review_bayes"] = bayes(M)

Global mean review (order-weighted): 4.12
Top 20 by review, m=10 vs m=20 overlap: 95%
Top 20 by review, m=40 vs m=20 overlap: 80%


## 4 · Percentiles and score

In [17]:
METRICS = {   # mart column: (short name, higher is better)
    "dispatch_on_time_rate":  ("dispatch_on_time", True),
    "dispatch_lead_p90_days": ("lead_p90", False),
    "review_bayes":           ("review_bayes", True),
    "negative_review_rate":   ("negative_reviews", False),
    "cancel_rate":            ("cancel", False),
    "freight_ratio":          ("freight_ratio", False),
}
small = el.groupby("main_category")["seller_id"].transform("count") < 5
for col, (short, hib) in METRICS.items():
    within = el.groupby("main_category")[col].rank(pct=True, ascending=hib)
    glob = el[col].rank(pct=True, ascending=hib)
    el[f"pct_{short}"] = 100 * within.where(~small, glob)

PCT = [f"pct_{s}" for s in ORDER]
missing = el[PCT].isna().any(axis=1)
print(f"Sellers using the global percentile (small category): {small.sum()}")
print(f"Sellers with at least one metric missing (set to 50): {missing.sum()}")
print(el[PCT].isna().sum().to_string())
el[PCT] = el[PCT].fillna(50)

def score(weights):
    return sum(el[f"pct_{k}"] * w for k, w in weights.items())

el["score_default"] = score(W)

Sellers using the global percentile (small category): 52
Sellers with at least one metric missing (set to 50): 0
pct_dispatch_on_time    0
pct_lead_p90            0
pct_review_bayes        0
pct_negative_reviews    0
pct_cancel              0
pct_freight_ratio       0


**Tiers are relative, not fixed.** A score built from percentiles centres around 50 by construction, so fixed cuts at 60 and 80 would put most sellers in tier C. Tiers use the score distribution instead: **A = top 20%, C = bottom 20%, B = the rest.** The cut values are exported so the dashboard reproduces them.

In [18]:
cut_a, cut_c = el["score_default"].quantile(0.80), el["score_default"].quantile(0.20)
el["tier"] = np.where(el["score_default"] >= cut_a, "A", np.where(el["score_default"] < cut_c, "C", "B"))
print(f"Tier cuts: A >= {cut_a:.1f} · C < {cut_c:.1f}")
print(el["tier"].value_counts().sort_index().to_string())

fig = go.Figure()
for t, color in [("C", C["unfavorable"]), ("B", C["sage_500"]), ("A", C["forest_700"])]:
    fig.add_histogram(x=el.loc[el["tier"] == t, "score_default"], name=f"Tier {t}",
                      xbins=dict(start=0, end=100, size=2.5), marker_color=color)
for v, label in [(cut_c, "C | B"), (cut_a, "B | A")]:
    fig.add_vline(x=v, line=dict(color=C["ink"], width=1, dash="dash"))
    fig.add_annotation(x=v, y=1, yref="paper", text=f"{label} · {v:.0f}", showarrow=False,
                       xanchor="left", xshift=4, yanchor="top", font=dict(family=MONO, size=10, color=C["ink"]))
fig.update_layout(barmode="stack", bargap=0.05)
fig.update_xaxes(title="Supplier score (0 to 100, weighted percentile within category)", range=[0, 100])
fig.update_yaxes(title="Sellers")
finish(fig, f"{len(el):,} eligible sellers scored; tiers split the top and bottom 20%",
       f"Score distribution · weights: dispatch {W['dispatch_on_time']:.0%}, review {W['review_bayes']:.0%}, "
       f"freight {W['freight_ratio']:.0%}, lead time {W['lead_p90']:.0%}, cancel {W['cancel']:.0%}, negative {W['negative_reviews']:.0%}")
fig.show()
save(fig, "08_score_distribution")

Tier cuts: A >= 70.9 · C < 33.4
tier
A    127
B    379
C    127


WindowsPath('C:/dev/procurement-analytics-olist/reports/figures/08_score_distribution.png')

## 5 · Sensitivity to the weights
If the ranking changes a lot when the weights change, the weights drive the conclusion and the dashboard must let the user set them. Threshold set in advance: top-20 overlap below 60% is reported as a finding.

In [19]:
SETS = {
    "default": W,
    "delivery_first": dict(zip(ORDER, [.45, .15, .15, .05, .05, .15])),
    "quality_first":  dict(zip(ORDER, [.20, .05, .40, .15, .05, .15])),
    "equal":          dict(zip(ORDER, [1 / 6] * 6)),
}
for name, w in SETS.items():
    el[f"score_{name}"] = score(w)

top20 = lambda s: set(el.nlargest(20, s)["seller_id"])
bot20 = lambda s: set(el.nsmallest(20, s)["seller_id"])
rows = []
for name in list(SETS)[1:]:
    rows.append({"weights": name,
                 "spearman_vs_default": spearmanr(el["score_default"], el[f"score_{name}"])[0],
                 "top20_overlap": len(top20("score_default") & top20(f"score_{name}")) / 20,
                 "bottom20_overlap": len(bot20("score_default") & bot20(f"score_{name}")) / 20})
sens = pd.DataFrame(rows)
sens.style.format({"spearman_vs_default": "{:.2f}", "top20_overlap": "{:.0%}", "bottom20_overlap": "{:.0%}"}).hide(axis="index")

weights,spearman_vs_default,top20_overlap,bottom20_overlap
delivery_first,0.97,85%,85%
quality_first,0.95,70%,80%
equal,0.97,85%,60%


## 6 · Action matrix
Spend percentile vs score, cut at the median of each among eligible sellers. **Escalate** = high spend, low score: the sellers where poor performance costs most.

In [20]:
el["spend_pct"] = 100 * el["spend_brl"].rank(pct=True)
hi_spend = el["spend_pct"] >= 50
hi_score = el["score_default"] >= el["score_default"].median()
el["action_quadrant"] = np.select(
    [hi_spend & ~hi_score, hi_spend & hi_score, ~hi_spend & ~hi_score],
    ["Escalate", "Protect", "Monitor"], default="Maintain")

# dominant reason: metric with the largest weighted gap to a perfect percentile
gaps = pd.DataFrame({k: W[k] * (100 - el[f"pct_{k}"]) for k in ORDER})
el["main_driver"] = gaps.idxmax(axis=1)

quad = (el.groupby("action_quadrant")
          .agg(sellers=("seller_id", "count"), spend_brl=("spend_brl", "sum"))
          .assign(share_of_spend=lambda d: d["spend_brl"] / el["spend_brl"].sum()))
print(quad.to_string())
esc = el[el["action_quadrant"] == "Escalate"]
print("\nMain driver among Escalate sellers:")
print(esc["main_driver"].value_counts(normalize=True).map("{:.0%}".format).to_string())

                 sellers     spend_brl  share_of_spend
action_quadrant                                       
Escalate             160 4,616,650.120           0.445
Maintain             160   728,045.840           0.070
Monitor              156   680,821.970           0.066
Protect              157 4,347,254.930           0.419

Main driver among Escalate sellers:
main_driver
dispatch_on_time    71%
review_bayes        29%


**From quadrant to shortlist.** A median split always puts about a quarter of sellers in Escalate, and a score just below the median is average, not poor. The action list is therefore narrowed to **Escalate sellers that are also tier C** (bottom 20% of scores): high spend and clearly poor performance.

In [21]:
el["priority"] = (el["action_quadrant"] == "Escalate") & (el["tier"] == "C")
pr = el[el["priority"]]
pr_share = pr["spend_brl"].sum() / el["spend_brl"].sum()
print(f"Priority sellers (Escalate and tier C): {len(pr)} of {len(esc)} Escalate sellers")
print(f"Share of eligible spend:                {pr_share:.1%}")
print("\nMain driver among priority sellers:")
print(pr["main_driver"].value_counts(normalize=True).map("{:.0%}".format).to_string())
print("\nMedian raw metrics, priority vs all eligible:")
cmp = pd.DataFrame({"priority": pr[["dispatch_on_time_rate", "review_bayes", "dispatch_lead_p90_days", "freight_ratio"]].median(),
                    "all_eligible": el[["dispatch_on_time_rate", "review_bayes", "dispatch_lead_p90_days", "freight_ratio"]].median()})
print(cmp.to_string())

Priority sellers (Escalate and tier C): 60 of 160 Escalate sellers
Share of eligible spend:                14.7%

Main driver among priority sellers:
main_driver
dispatch_on_time    80%
review_bayes        20%

Median raw metrics, priority vs all eligible:
                        priority  all_eligible
dispatch_on_time_rate      0.777         0.950
review_bayes               3.889         4.168
dispatch_lead_p90_days     8.630         4.307
freight_ratio              0.182         0.210


In [22]:
fig = go.Figure()
layers = [("Maintain", el["action_quadrant"] == "Maintain", C["context"], 7),
          ("Monitor", el["action_quadrant"] == "Monitor", C["context"], 7),
          ("Protect", el["action_quadrant"] == "Protect", C["forest_700"], 8),
          ("Escalate, tier B", (el["action_quadrant"] == "Escalate") & ~el["priority"], "#D08470", 8),
          ("Escalate, tier C · priority", el["priority"], C["unfavorable"], 10)]
for name, mask, color, size in layers:
    d = el[mask]
    fig.add_scatter(x=d["spend_pct"], y=d["score_default"], mode="markers", name=name,
                    marker=dict(symbol="square", size=size, color=color),
                    customdata=np.stack([d["main_category"], d["spend_brl"], d["main_driver"]], axis=-1),
                    hovertemplate="%{customdata[0]}<br>BRL %{customdata[1]:,.0f} · score %{y:.0f}"
                                  "<br>main gap: %{customdata[2]}<extra></extra>")
med = el["score_default"].median()
fig.add_shape(type="line", x0=50, x1=50, y0=0, y1=100, line=dict(color=C["baseline"], width=1, dash="dash"))
fig.add_shape(type="line", x0=0, x1=100, y0=med, y1=med, line=dict(color=C["baseline"], width=1, dash="dash"))
for x, y, t in [(52, 2, "ESCALATE"), (52, 98, "PROTECT"), (2, 2, "MONITOR"), (2, 98, "MAINTAIN")]:
    fig.add_annotation(x=x, y=y, text=t, showarrow=False, xanchor="left",
                       yanchor="bottom" if y < 50 else "top", font=dict(family=MONO, size=10, color=C["axis"]))
fig.add_shape(type="line", x0=50, x1=100, y0=cut_c, y1=cut_c, line=dict(color=C["unfavorable"], width=1, dash="dot"))
fig.add_annotation(x=100, y=cut_c, text=f"TIER C · BELOW {cut_c:.0f}", showarrow=False, xanchor="right",
                   yanchor="bottom", font=dict(family=MONO, size=10, color=C["unfavorable"]))
fig.update_xaxes(title="Spend percentile among eligible sellers", range=[0, 100])
fig.update_yaxes(title="Supplier score", range=[0, 100], zeroline=False)
esc_share = quad.loc["Escalate", "share_of_spend"] if "Escalate" in quad.index else 0
finish(fig, f"{len(pr)} high-spend sellers in the bottom tier hold {pr_share:.0%} of eligible spend",
       "Action matrix · spend percentile vs supplier score · quadrant cuts at the median · priority = Escalate and tier C")
fig.show()
save(fig, "09_action_matrix")

WindowsPath('C:/dev/procurement-analytics-olist/reports/figures/09_action_matrix.png')

## 7 · Explainability check
A score is only usable in a supplier meeting if each point can be traced. One seller per tier, broken down into the contribution of each metric.

In [23]:
pick = pd.concat([el[el["tier"] == t].sort_values("score_default").iloc[[len(el[el["tier"] == t]) // 2]]
                  for t in ["A", "B", "C"]])
contrib = pd.DataFrame({k: pick[f"pct_{k}"] * W[k] for k in ORDER})
contrib["score"] = contrib.sum(axis=1)
contrib.index = [f"{t} · {s[:8]}" for t, s in zip(pick["tier"], pick["seller_id"])]
contrib.round(1)

,dispatch_on_time,lead_p90,review_bayes,negative_reviews,cancel,freight_ratio,score
A · 116ccb1a,30.500,9.300,30.000,4.800,3.000,0.600,78.100
B · 6a8b085f,22.000,5.600,15.600,2.600,3.000,2.800,51.500
C · 6a0cbc8a,1.100,1.400,7.100,2.200,3.200,7.900,23.000


## 8 · Export (contract consumed by Tableau and Hugging Face)

In [24]:
RAW = ["dispatch_on_time_rate", "dispatch_lead_p90_days", "review_mean", "review_n", "review_bayes",
       "negative_review_rate", "cancel_rate", "freight_ratio", "customer_on_time_rate"]
cols = (["seller_id", "seller_state", "main_category", "eligible", "n_orders", "spend_brl", "freight_brl"]
        + RAW + PCT + ["score_default", "tier", "action_quadrant", "priority", "main_driver", "spend_pct"])

new_cols = [c for c in cols if c not in sc.columns]
out = sc.merge(el[["seller_id"] + new_cols], on="seller_id", how="left")
out = out[cols].sort_values("score_default", ascending=False, na_position="last")

PROC = ROOT / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)
out.to_csv(PROC / "scorecard_sellers.csv", index=False)
out.to_parquet(PROC / "scorecard_sellers.parquet", index=False)
pd.DataFrame([{"metric": k, "weight": v} for k, v in W.items()]
             + [{"metric": "tier_a_min", "weight": cut_a}, {"metric": "tier_c_max", "weight": cut_c},
                {"metric": "review_shrinkage_m", "weight": M}, {"metric": "min_orders", "weight": MIN_ORDERS}]
             ).to_csv(PROC / "scorecard_parameters.csv", index=False)

print(f"scorecard_sellers: {len(out):,} rows ({out['eligible'].sum():,} scored), {len(cols)} columns")
print("scorecard_parameters.csv written")

scorecard_sellers: 3,068 rows (633 scored), 28 columns
scorecard_parameters.csv written


## 9 · Findings for `docs/findings.md`

In [25]:
s0 = sens.set_index("weights")
min_overlap = s0["top20_overlap"].min()
findings = [
    f"**{len(el):,} sellers (30+ orders) cover {coverage:.1%} of spend and are scored.** "
    f"Cancellation weight {'was reduced to 5%' if W['cancel'] == .05 else 'stayed at 15%'}: "
    f"{zero_cancel:.0%} of eligible sellers have zero cancellations.",
    f"**{len(pr)} sellers are the action shortlist:** high spend and bottom-tier score, holding {pr_share:.1%} of eligible spend; "
    f"their most common gap is {pr['main_driver'].value_counts().index[0].replace('_', ' ')} "
    f"({pr['main_driver'].value_counts(normalize=True).iloc[0]:.0%} of them).",
    (f"**The ranking is robust to the weights.** " if min_overlap >= 0.60 else
     f"**The ranking depends on the weights.** ") +
    f"Across three alternative weightings, Spearman vs default is {s0['spearman_vs_default'].min():.2f} or higher "
    f"and top-20 overlap is at least {min_overlap:.0%}.",
]
for f in findings:
    print("- " + f + "\n")

- **633 sellers (30+ orders) cover 77.1% of spend and are scored.** Cancellation weight was reduced to 5%: 77% of eligible sellers have zero cancellations.

- **60 sellers are the action shortlist:** high spend and bottom-tier score, holding 14.7% of eligible spend; their most common gap is dispatch on time (80% of them).

- **The ranking is robust to the weights.** Across three alternative weightings, Spearman vs default is 0.95 or higher and top-20 overlap is at least 70%.

